# **1.  you are developing a technical documentation search system. user search for error messages such as:**
# **2.  to implement a keyword-based retrieval system that ranks document according to the number of query keywords found in each document**


In [2]:
documents = [
    "HTTP 401 Unauthorized occurs when the API access token is missing or invalid.",
    "HTTP 500 Internal Server Error indicates a problem on the server side.",
    "API errors can be debugged using application logs.",
    "Azure API authentication requires a valid bearer token in the Authorization header.",
    "HTTP 403 Forbidden means the user is authenticated but does not have permission."
]

query = "HTTP 401 AUTHENTICATION ERROR"

query_words = set(query.lower().split())

results = []

for document in documents:
    document_words = set(document.lower().split())
    common_words = query_words.intersection(document_words)
    score = len(common_words)
    results.append((document, score))

results = sorted(results, key=lambda x:x[1], reverse=True)
for doc, score in results:
    print(f"Score: {score} | Document: {doc}")

Score: 2 | Document: HTTP 401 Unauthorized occurs when the API access token is missing or invalid.
Score: 2 | Document: HTTP 500 Internal Server Error indicates a problem on the server side.
Score: 1 | Document: Azure API authentication requires a valid bearer token in the Authorization header.
Score: 1 | Document: HTTP 403 Forbidden means the user is authenticated but does not have permission.
Score: 0 | Document: API errors can be debugged using application logs.


In [15]:
import google.generativeai as genai
from google.colab import userdata
api_key=userdata.get("Gemini_api_key")
genai.configure(api_key=api_key)

In [16]:
import numpy as np
from sklearn.metrics.pairwise import cosine_similarity

math_documents = [
    "Calculus is very important part of mathematics",
    "Linear Algebra includes vectors, matrix, polynomials etc",
    "Mathematics 1 is a simple subject",
    "Proficiency in statistics and probability is needed for data science",
    "Monthly 20 classes are scheduled for mathematics"
]
query="What to study in mathematics?"

# 1. Embed documents
document_embeddings = []
for doc_text in math_documents:
  res = genai.embed_content(
      model="gemini-embedding-001",
      content=doc_text
  )
  document_embeddings.append(res['embedding'])

# Convert to numpy array for cosine similarity
document_embeddings_np = np.array(document_embeddings)

# 2. Embed query
query_embedding_res = genai.embed_content(
    model="gemini-embedding-001",
    content=query
)
query_embedding = np.array(query_embedding_res['embedding']).reshape(1, -1)

# 3. Calculate vector similarity (VS)
vector_similarities = cosine_similarity(
    query_embedding,
    document_embeddings_np
)[0]

# 4. Keyword-based scoring (KS)
query_words = set(query.lower().split())
keyword_scores = []

for doc_text in math_documents:
    doc_words = set(doc_text.lower().split())
    common_words = query_words.intersection(doc_words)
    score = len(common_words)
    keyword_scores.append(score)

# Normalize keyword scores
max_keyword_score = max(keyword_scores) if keyword_scores else 0
normalized_keyword_scores = [score / max_keyword_score if max_keyword_score > 0 else 0 for score in keyword_scores]

# 5. Hybrid Score (FS) - weighted combination
hybrid_scores = []
for vs, nks in zip(vector_similarities, normalized_keyword_scores):
  # Adjust weights as needed
  final_score = (vs * 0.6) + (nks * 0.4)
  hybrid_scores.append(final_score)

# Rank documents by hybrid score
ranks = sorted(zip(math_documents, hybrid_scores), key=lambda x: x[1], reverse=True)

print("Top hybrid search results:")
for rank, (doc, score) in enumerate(ranks[:3], start=1):
  print(f"Rank: {rank}, Document: {doc}, Score: {score:.4f}")

Top hybrid search results:
Rank: 1, Document: Proficiency in statistics and probability is needed for data science, Score: 0.7624
Rank: 2, Document: Mathematics 1 is a simple subject, Score: 0.4156
Rank: 3, Document: Monthly 20 classes are scheduled for mathematics, Score: 0.3961
